# Phase 2 Notebook: Candidate Generation
Prepares data extracted from data/10_mention_detection for candidate generation.

## Agenda
### Step 1: Load Setup context
1. Load Classes data/00_setup/classes.csv
2. Load Properties data/00_setup/properties.csv
* Load Broadcasting Programs data/00_setup/broadcasting_programs.csv

### Step 2: Load Phase 1 Output
1. Load Seasons data/10_mention_detection/seasons.csv
2. Load Episodes data/10_mention_detection/episodes.csv
3. Load Publications data/10_mention_detection/publications.csv
4. Load Persons data/10_mention_detection/persons.csv
5. (Currently not well supported, concept only: Load Topics data/10_mention_detection/topics.csv)
Reduce to required context

### Step 3: Create bare minimum lookup references:
#### Seasons
Reduce to the following columns: 
* season_id
* season_label
* start_time
* end_time
* episode_count

Add the following information by splitting the season label at separators, such as comma:
* season_label_chunk1
* season_label_chunk2
* season_label_chunk3
* ...

#### Episodes
Reduce to the following columns:
* episode_id
* sendungstitel
* season
* folge
* folgennr

of the information of staffel is contained in season, remove it. if not, print the episode as an output and print both staffel and season values. Most often, season will be "Staffel 1" and season will be "1", so it is usually redundant.

#### Publications
Append to the publication data to the episode data by looking up the episode_id and appending columns as follows:
* date -> publication_data_{publication_index}
* time -> publication_time_{publication_index}
* duration -> publication_duration_{publication_index}
* program -> publication_program_{publication_index}
* prod_nr_sendung -> publication_prod_nr_sendung_{publication_index}
* prod_nr_secondary -> publication_prod_nr_secondary_{publication_index}

#### Persons
Reduce to the following columns:
* mention_id
* episode_id
* name
* beschreibung

Then add a name_cleaned column by making every Uppercase letter that is not at the beginning of a word lowercase, (i.e., Elmar THEVEßEN -> Elmar Theveßen; Robin ALEXANDER -> Robin Alexander)
* name_cleaned     

Append to the publication data to the episode data by looking up the episode_id and appending columns as follows:
* name_cleaned = guest_{counter}
* beschreibung = guest_{counter}_description
so that every episode has the columns guest_0, guest_0_description, guest_1, guest_1_description, etc.

#### Topic
Append to the publication data to the episode data by looking up the episode_id and appending columns as follows:
* topic = topic_{counter starting at 0 per episode}
so that every episode has the columns topic_0, topic_1, etc.


## Step 1

In [ ]:
from process.bootstrap import bootstrap_notebook_paths

ROOT, SRC = bootstrap_notebook_paths()

from process.candidate_generation.broadcasting_program import load_setup_context
from process.candidate_generation.persistence import phase2_output_dir, persist_setup_outputs

PHASE1_DIR = ROOT / "data" / "10_mention_detection"
PHASE2_DIR = phase2_output_dir(ROOT)

setup_ctx = load_setup_context(ROOT)
classes_df = setup_ctx["classes"]
properties_df = setup_ctx["properties"]
broadcasting_programs_df = setup_ctx["broadcasting_programs"]

paths = persist_setup_outputs(ROOT, classes_df, properties_df, broadcasting_programs_df)

print(f"ROOT: {ROOT}")
print(f"Phase 1 dir: {PHASE1_DIR}")
print(f"Phase 2 dir (ownership): {PHASE2_DIR}")

print("\nLoaded setup context:")
print(f"  classes: {len(classes_df)} rows -> {paths['classes'].name}")
print(f"  properties: {len(properties_df)} rows -> {paths['properties'].name}")
print(f"  broadcasting_programs: {len(broadcasting_programs_df)} rows -> {paths['broadcasting_programs'].name}")

display(classes_df.head(5))
display(properties_df.head(5))
display(broadcasting_programs_df.head(5))

## Step 2

In [ ]:
from process.candidate_generation.season import load_seasons_context
from process.candidate_generation.episode import load_episodes_context, load_publications_context
from process.candidate_generation.person import load_persons_context
from process.candidate_generation.topic import load_topics_context
from process.candidate_generation.persistence import persist_seasons_output, persist_episodes_output

seasons_ctx_df = load_seasons_context(ROOT)
episodes_ctx_df = load_episodes_context(ROOT)
publications_ctx_df = load_publications_context(ROOT)
persons_ctx_df = load_persons_context(ROOT)
topics_ctx_df = load_topics_context(ROOT)

# Persist only the key evolving dataframes.
seasons_path = persist_seasons_output(ROOT, seasons_ctx_df)
episodes_path = persist_episodes_output(ROOT, episodes_ctx_df)

print("Phase 1 input loaded and reduced to required context:")
print(f"  seasons_ctx_df: {seasons_ctx_df.shape} -> {seasons_path.name}")
print(f"  episodes_ctx_df: {episodes_ctx_df.shape} -> {episodes_path.name}")
print(f"  publications_ctx_df: {publications_ctx_df.shape}")
print(f"  persons_ctx_df: {persons_ctx_df.shape}")
print(f"  topics_ctx_df: {topics_ctx_df.shape}  (concept-only support)")

display(seasons_ctx_df.head(5))
display(episodes_ctx_df.head(5))
display(publications_ctx_df.head(5))
display(persons_ctx_df.head(5))
display(topics_ctx_df.head(5))

## Step 3

### Seasons

In [ ]:
from process.candidate_generation.season import build_seasons_lookup
from process.candidate_generation.persistence import persist_seasons_output

seasons_lookup_df = build_seasons_lookup(seasons_ctx_df)
seasons_path = persist_seasons_output(ROOT, seasons_lookup_df)

print(f"seasons_lookup_df: {seasons_lookup_df.shape} -> {seasons_path.name}")
display(seasons_lookup_df.head(10))

### Episodes

In [ ]:
from process.candidate_generation.episode import build_episodes_lookup
from process.candidate_generation.persistence import persist_episodes_output

episodes_lookup_df, season_staffel_mismatches_df = build_episodes_lookup(episodes_ctx_df)
episodes_path = persist_episodes_output(ROOT, episodes_lookup_df)

print(f"Rows with non-redundant staffel/season values: {len(season_staffel_mismatches_df)}")
if not season_staffel_mismatches_df.empty:
    display(season_staffel_mismatches_df.head(25))

print(f"episodes_lookup_df after reduction: {episodes_lookup_df.shape} -> {episodes_path.name}")
display(episodes_lookup_df.head(10))

### Publications

In [ ]:
from process.candidate_generation.episode import append_publications_to_episodes
from process.candidate_generation.persistence import persist_episodes_output

episodes_lookup_df, pub_long_df, pub_wide_df = append_publications_to_episodes(
    episodes_lookup_df, publications_ctx_df
)
episodes_path = persist_episodes_output(ROOT, episodes_lookup_df)

print(f"Publication rows transformed: {len(pub_long_df)}")
print(f"Publication columns appended: {len(pub_wide_df.columns) - 1}")
print(f"episodes_lookup_df after publications merge: {episodes_lookup_df.shape} -> {episodes_path.name}")
display(episodes_lookup_df.head(10))

### Persons

In [ ]:
import importlib
import process.candidate_generation.person as person_mod
import process.candidate_generation.persistence as persistence_mod

importlib.reload(person_mod)
importlib.reload(persistence_mod)

episodes_lookup_df, persons_lookup_df, guest_long_df, guest_wide_df, duplicate_guests_df = person_mod.append_persons_to_episodes(
    episodes_lookup_df, persons_ctx_df
)
episodes_path = persistence_mod.persist_episodes_output(ROOT, episodes_lookup_df)
duplicates_path = persistence_mod.persist_guest_duplicate_feedback(ROOT, duplicate_guests_df)

print(f"persons_lookup_df (after de-duplication): {persons_lookup_df.shape}")
print(f"guest duplicate feedback rows: {len(duplicate_guests_df)} -> {duplicates_path.name}")
print(f"guest columns appended: {len(guest_wide_df.columns) - 1}")
print(f"episodes_lookup_df after guests merge: {episodes_lookup_df.shape} -> {episodes_path.name}")
display(persons_lookup_df.head(10))
if not duplicate_guests_df.empty:
    display(duplicate_guests_df.head(10))
display(episodes_lookup_df.head(10))

### Topics

In [ ]:
from process.candidate_generation.topic import append_topics_to_episodes
from process.candidate_generation.persistence import persist_episodes_output

episodes_lookup_df, topics_lookup_df, topic_long_df, topic_wide_df = append_topics_to_episodes(
    episodes_lookup_df, topics_ctx_df
)
episodes_path = persist_episodes_output(ROOT, episodes_lookup_df)

print(f"topics_lookup_df: {topics_lookup_df.shape} (concept-only support)")
print(f"topic columns appended: {len(topic_wide_df.columns) - 1}")
print(f"episodes_lookup_df final shape: {episodes_lookup_df.shape} -> {episodes_path.name}")
display(topics_lookup_df.head(10))
display(episodes_lookup_df)

## Proceed to search in dedicated notebooks (one per Source)
The preparation for Phase 2 is now done. Proceed with executing the Notebooks, preferably in this order:
1. Wikibase
2. Wikidata
3. Fernsehserien_de
4. Any other